In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from melapa.OmicsDataset import OmicsDataset
from melapa.helper_plot import boxplot_pca_significance

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
df_genes = pd.read_csv("path/to/AstroBF_fastq/RN22346/230106_A01366_0335_BHJJHNDSX5/genes_preprocessing/df_genes_filtered_normalized.csv", index_col=0)

In [ ]:
metadata_path = "path/to/AstroBF_fastq/RN22346/RN22346.xlsx"
metadata = pd.read_excel(metadata_path)
metadata['category'] = metadata["Sample Genotype"] + "_" + metadata["Sample Treatment"]

df_genes.columns = metadata.reset_index().set_index("limsid").loc[df_genes.columns]['category'].values

In [ ]:
treatments = {i: [i] for i in df_genes.columns}

In [ ]:
treatments = dict(sorted(treatments.items()))
treatments

In [ ]:
genes_dataset = OmicsDataset(df=df_genes, treatments=treatments, 
                             treatment_name='category', feature_type='gene', nb_comp=17)

In [ ]:
palette = [
    (0.6, 0.0, 0.3),   # dark pink
    (0.9, 0.4, 0.7),   # medium pink
    (1.0, 0.8, 0.9),   # light pink
    (0.0, 0.2, 0.5),   # dark blue
    (0.2, 0.5, 0.9),   # medium blue
    (0.7, 0.8, 1.0),   # light blue
    (0.0, 0.4, 0.2),   # dark green
    (0.3, 0.7, 0.5),   # medium green
    (0.8, 1.0, 0.9),   # light green
    (0.4, 0.2, 0.0),   # dark orange
    (0.9, 0.6, 0.2),   # medium orange
    (1.0, 0.9, 0.7),    # light orange
    (0.4, 0.0, 0.6),   # dark purple
    (0.7, 0.5, 0.9),   # medium purple
    (0.9, 0.8, 1.0),    # light purple
    (0.5, 0.3, 0.1),   # brown dark
    (0.7, 0.5, 0.3),   # brown medium
    (0.9, 0.8, 0.6),   # brown light
]

In [ ]:
genes_dataset.hierarchical_clustering(palette=palette, treatments=treatments, color_threshold=0)

In [ ]:
genes_dataset.perform_svd_analysis(top_n=18)
plt.xticks(rotation=90, fontsize=8);

In [ ]:
genes_dataset.pca['treatment'] = [idx.split('_')[1] for idx in genes_dataset.pca.index]
genes_dataset.pca['genotype'] = [idx.split('_')[0] for idx in genes_dataset.pca.index]


In [ ]:
genes_dataset.pca['mutation'] = genes_dataset.pca['genotype'].apply(lambda x: 'ctrl' if 'ctrl' in x else 'vcp')

In [ ]:
genes_dataset.pca

In [ ]:
palette_genotype = {"ctrl2": 'palegreen', "ctrl5": "limegreen", "ctrl6": "forestgreen", "mut1": "pink", "mut2": "hotpink", "mut4": "deeppink" }

In [ ]:
plt.figure(figsize=(20, 15))
for pc in range(17):
    plt.subplot(5, 4, pc+1)
    for genotype in genes_dataset.pca['genotype'].unique():
        subset = genes_dataset.pca[genes_dataset.pca['genotype'] == genotype]
        subset.index = [idx.split('_')[1] for idx in subset.index]
        subset = subset.loc[["basal", "A1 low", "A1 high"]]
        plt.scatter(x=subset.index, y=subset[f'PC{pc+1}'], label=genotype, color=palette_genotype[genotype])
        plt.plot(subset.index, subset[f'PC{pc+1}'], color=palette_genotype[genotype])
    if pc == 3:
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    sns.despine()
    plt.title(f"PC{pc+1}", weight='bold')
plt.tight_layout()

In [ ]:
boxplot_pca_significance(genes_dataset.pca, treatment_col='mutation', fontsize=10, palette={'ctrl': 'limegreen', 'vcp': 'hotpink'})

In [ ]:
genes_dataset.pca

In [ ]:
%%capture cap --no-display

from melapa.LMM_helper import lmer_per_PC_re

## See which component captures the difference sensitive/resistant

plt.figure(figsize=(10, 5))
## See which component captures the batch
anovas = []
p_vals = []

for i in range(17):
    PC = i+1
    anova, p_val = lmer_per_PC_re(genes_dataset.pca, random_effects=['treatment'], fixed_effect='mutation', PC=PC)
    anovas.append(anova)
    p_vals.append(p_val)

plt.subplot(1, 2, 1)
sns.barplot(x=np.arange(1, 18), y=-np.log10(p_vals), color='grey')
plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel('-log10(p-value)')
plt.title(f'Mutation \n ANOVA between full and reduced model \n p-value')

delta_deviances = []
for PC in range(len(anovas)):
    delta_deviance = anovas[PC].loc['MODEL2', 'deviance'] - anovas[PC].loc['MODEL1', 'deviance'] ## Difference between the full and reduced model
    delta_deviances.append(delta_deviance)

plt.subplot(1, 2, 2)
sns.barplot(x=[f"PC{i+1}" for i in range(len(anovas))], y=delta_deviances, color='grey')
#plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel(r'$\Delta$ deviance')
plt.title(f'Mutation \n ANOVA between full and reduced model \n Deviance')

plt.tight_layout()

In [ ]:
PCs = [f"PC{PC+1}" for PC in list(np.argwhere(np.array(p_vals) < 0.05).flatten())] + ['mutation']
sns.pairplot(genes_dataset.pca[PCs], hue='mutation',  palette={'ctrl': 'limegreen', 'vcp': 'hotpink'})

In [ ]:
boxplot_pca_significance(genes_dataset.pca, treatment_col='treatment', fontsize=10, palette={'basal': 'dimgray', 'A1 low': 'deepskyblue', "A1 high": "darkorange"})

In [ ]:
%%capture cap --no-display

from melapa.LMM_helper import lmer_per_PC_re

## See which component captures the difference sensitive/resistant

plt.figure(figsize=(10, 5))
## See which component captures the batch
anovas = []
p_vals = []

for i in range(17):
    PC = i+1
    anova, p_val = lmer_per_PC_re(genes_dataset.pca, random_effects=['mutation'], fixed_effect='treatment', PC=PC)
    anovas.append(anova)
    p_vals.append(p_val)

plt.subplot(1, 2, 1)
sns.barplot(x=np.arange(1, 18), y=-np.log10(p_vals), color='grey')
plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel('-log10(p-value)')
plt.title(f'Treatment \n ANOVA between full and reduced model \n p-value')

delta_deviances = []
for PC in range(len(anovas)):
    delta_deviance = anovas[PC].loc['MODEL2', 'deviance'] - anovas[PC].loc['MODEL1', 'deviance'] ## Difference between the full and reduced model
    delta_deviances.append(delta_deviance)

plt.subplot(1, 2, 2)
sns.barplot(x=[f"PC{i+1}" for i in range(len(anovas))], y=delta_deviances, color='grey')
#plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel(r'$\Delta$ deviance')
plt.title(f'Treatment \n ANOVA between full and reduced model \n Deviance')

plt.tight_layout()

In [ ]:
PCs = [f"PC{PC+1}" for PC in list(np.argwhere(np.array(p_vals) < 0.05).flatten())] + ['treatment']
sns.pairplot(genes_dataset.pca[PCs], hue='treatment',  palette={'basal': 'dimgray', 'A1 low': 'deepskyblue', "A1 high": "darkorange"})

In [ ]:
boxplot_pca_significance(genes_dataset.pca, treatment_col='genotype', fontsize=10, palette=palette_genotype)

In [ ]:
%%capture cap --no-display

from melapa.LMM_helper import lmer_per_PC_re

## See which component captures the difference sensitive/resistant

plt.figure(figsize=(10, 5))
## See which component captures the batch
anovas = []
p_vals = []

for i in range(17):
    PC = i+1
    anova, p_val = lmer_per_PC_re(genes_dataset.pca, random_effects=['treatment'], fixed_effect='genotype', PC=PC)
    anovas.append(anova)
    p_vals.append(p_val)

plt.subplot(1, 2, 1)
sns.barplot(x=np.arange(1, 18), y=-np.log10(p_vals), color='grey')
plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel('-log10(p-value)')
plt.title(f'Genotype \n ANOVA between full and reduced model \n p-value')

delta_deviances = []
for PC in range(len(anovas)):
    delta_deviance = anovas[PC].loc['MODEL2', 'deviance'] - anovas[PC].loc['MODEL1', 'deviance'] ## Difference between the full and reduced model
    delta_deviances.append(delta_deviance)

plt.subplot(1, 2, 2)
sns.barplot(x=[f"PC{i+1}" for i in range(len(anovas))], y=delta_deviances, color='grey')
#plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel(r'$\Delta$ deviance')
plt.title(f'Genotype \n ANOVA between full and reduced model \n Deviance')

plt.tight_layout()

In [ ]:
PCs = [f"PC{PC+1}" for PC in list(np.argwhere(np.array(p_vals) < 0.05).flatten())] + ['genotype']
sns.pairplot(genes_dataset.pca[PCs], hue='genotype',  palette=palette_genotype)

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1,2,1)
sns.scatterplot(data=genes_dataset.pca, x='PC1', y='PC5', hue='mutation', palette={'ctrl': 'limegreen', 'vcp': 'hotpink'})
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.title("Colored by mutation", weight='bold', fontsize=13)
sns.despine()



plt.subplot(1,2,2)
sns.scatterplot(data=genes_dataset.pca, x='PC1', y='PC7', hue='treatment', palette={'basal': 'dimgray', 'A1 low': 'deepskyblue', "A1 high": "darkorange"})
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.title("Colored by treatment", weight='bold', fontsize=13)
sns.despine()


plt.suptitle("PCA on gene expression", weight='bold')
plt.tight_layout()


In summary:
- PC1 captures untreated vs treated
- PC5 captures the mutation
- PC7 captures the difference between A1 high and A1 low

## Pathway analysis

In [ ]:
import stringdb
from pybpanalysis.stringdb_helper import stringdb_barplot_one_class



In [ ]:
pairs_treated_PC1, pairs_untreated_PC1 = genes_dataset.projection_correlation_one_PC(PC=1, 
                                                                                     min_extreme='treated', 
                                                                                     max_extreme='untreated', 
                                                                                     k_clusters=12, 
                                                                                     min_extreme_color="purple", 
                                                                                     max_extreme_color="black", 
                                                                                     sort_by='projection', 
                                                                                     scaling=False)

sns.despine()
plt.title('PC1', weight='bold')
print(f"Number of pairs specific to the treated extreme: {len(pairs_treated_PC1)}")
print(f"Number of pairs specific to the untreated extreme: {len(pairs_untreated_PC1)}")

In [ ]:
from matplotlib import cm
string_ids = stringdb.get_string_ids([pair.split('/')[0][:-2] for pair in pairs_treated_PC1])
enrichment_df_PUD_PC3_acquired = stringdb.get_enrichment(string_ids.queryItem)
enrichment_df_PUD_PC3_acquired = enrichment_df_PUD_PC3_acquired[enrichment_df_PUD_PC3_acquired['number_of_genes'] >= 5]
if 'Process' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='Process', cmap=cm.cool, title='Up in treated samples (PC1)', vmin=0, vmax=0.5, frac_x=False)
if 'KEGG' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='KEGG', cmap=cm.cool, title='Up in treated samples (PC1)', vmin=0, vmax=0.5, frac_x=False)
if 'WikiPathways' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='WikiPathways', cmap=cm.cool, title='Up in treated samples (PC1)', vmin=0, vmax=0.5, frac_x=False)
if 'Keyword' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:

    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='Keyword', cmap=cm.cool, title='Up in treated samples (PC1)', vmin=0, vmax=0.5, frac_x=False)

from matplotlib import cm
string_ids = stringdb.get_string_ids([pair.split('/')[0][:-2] for pair in pairs_untreated_PC1])
enrichment_df_PUD_PC3_innate = stringdb.get_enrichment(string_ids.queryItem)
enrichment_df_PUD_PC3_innate = enrichment_df_PUD_PC3_innate[enrichment_df_PUD_PC3_innate['number_of_genes'] >= 5]
if 'Process' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='Process', cmap=cm.cool, title='Up in untreated samples (PC1)', vmin=0, vmax=0.5, frac_x=False)
if 'KEGG' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='KEGG', cmap=cm.cool, title='Up in untreated samples (PC1)', vmin=0, vmax=0.5, frac_x=False)
if 'WikiPathways' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='WikiPathways', cmap=cm.cool, title='Up in untreated samples (PC1)', vmin=0, vmax=0.5, frac_x=False)
if 'Keyword' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='Keyword', cmap=cm.cool, title='Up in untreated samples (PC1)', vmin=0, vmax=0.5, frac_x=False)

In [ ]:
pairs_vcp_PC5, pairs_ctrl_PC5 = genes_dataset.projection_correlation_one_PC(PC=5, 
                                                                                     min_extreme='VCP', 
                                                                                     max_extreme='CTRL', 
                                                                                     k_clusters=12, 
                                                                                     min_extreme_color="hotpink", 
                                                                                     max_extreme_color="limegreen", 
                                                                                     sort_by='projection', 
                                                                                     scaling=False)

sns.despine()
plt.title('PC5', weight='bold')
print(f"Number of pairs specific to the VCP extreme: {len(pairs_vcp_PC5)}")
print(f"Number of pairs specific to the CTRL extreme: {len(pairs_ctrl_PC5)}")


In [ ]:
from matplotlib import cm
string_ids = stringdb.get_string_ids([pair.split('/')[0][:-2] for pair in pairs_vcp_PC5])
enrichment_df_PUD_PC3_acquired = stringdb.get_enrichment(string_ids.queryItem)
enrichment_df_PUD_PC3_acquired = enrichment_df_PUD_PC3_acquired[enrichment_df_PUD_PC3_acquired['number_of_genes'] >= 5]
if 'Process' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='Process', cmap=cm.cool, title='Up in VCP samples (PC5)', vmin=0, vmax=0.5, frac_x=False)
if 'KEGG' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='KEGG', cmap=cm.cool, title='Up in VCP samples (PC5)', vmin=0, vmax=0.5, frac_x=False)
if 'WikiPathways' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='WikiPathways', cmap=cm.cool, title='Up in VCP samples (PC5)', vmin=0, vmax=0.5, frac_x=False)
if 'Keyword' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:

    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='Keyword', cmap=cm.cool, title='Up in VCP samples (PC5)', vmin=0, vmax=0.5, frac_x=False)

from matplotlib import cm
string_ids = stringdb.get_string_ids([pair.split('/')[0][:-2] for pair in pairs_ctrl_PC5])
enrichment_df_PUD_PC3_innate = stringdb.get_enrichment(string_ids.queryItem)
enrichment_df_PUD_PC3_innate = enrichment_df_PUD_PC3_innate[enrichment_df_PUD_PC3_innate['number_of_genes'] >= 5]
if 'Process' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='Process', cmap=cm.cool, title='Up in CTRL samples (PC5)', vmin=0, vmax=0.5, frac_x=False)
if 'KEGG' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='KEGG', cmap=cm.cool, title='Up in CTRL samples (PC5)', vmin=0, vmax=0.5, frac_x=False)
if 'WikiPathways' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='WikiPathways', cmap=cm.cool, title='Up in CTRL samples (PC5)', vmin=0, vmax=0.5, frac_x=False)
if 'Keyword' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='Keyword', cmap=cm.cool, title='Up in CTRL samples (PC5)', vmin=0, vmax=0.5, frac_x=False)

In [ ]:
pairs_a1_high_PC7, pairs_a1_low_PC7 = genes_dataset.projection_correlation_one_PC(PC=7, 
                                                                                     min_extreme='A1 high', 
                                                                                     max_extreme='A1 low', 
                                                                                     k_clusters=12, 
                                                                                     min_extreme_color="darkorange", 
                                                                                     max_extreme_color="deepskyblue", 
                                                                                     sort_by='correlation', 
                                                                                     scaling=False)

sns.despine()
plt.title('PC7', weight='bold')
print(f"Number of pairs specific to the A1 high extreme: {len(pairs_a1_high_PC7)}")
print(f"Number of pairs specific to the A1 low extreme: {len(pairs_a1_low_PC7)}")

In [ ]:
from matplotlib import cm
string_ids = stringdb.get_string_ids([pair.split('/')[0][:-2] for pair in pairs_a1_low_PC7])
enrichment_df_PUD_PC3_acquired = stringdb.get_enrichment(string_ids.queryItem)
enrichment_df_PUD_PC3_acquired = enrichment_df_PUD_PC3_acquired[enrichment_df_PUD_PC3_acquired['number_of_genes'] >= 5]
if 'Process' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='Process', cmap=cm.cool, title='Up in A1 low samples (PC7)', vmin=0, vmax=0.5, frac_x=False)
if 'KEGG' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='KEGG', cmap=cm.cool, title='Up in A1 low samples (PC7)', vmin=0, vmax=0.5, frac_x=False)
if 'WikiPathways' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='WikiPathways', cmap=cm.cool, title='Up in A1 low samples (PC7)', vmin=0, vmax=0.5, frac_x=False)
if 'Keyword' in enrichment_df_PUD_PC3_acquired.value_counts('category').index:

    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_acquired, nb_top=10, clas='Keyword', cmap=cm.cool, title='Up in A1 low samples (PC7)', vmin=0, vmax=0.5, frac_x=False)

from matplotlib import cm
string_ids = stringdb.get_string_ids([pair.split('/')[0][:-2] for pair in pairs_a1_high_PC7])
enrichment_df_PUD_PC3_innate = stringdb.get_enrichment(string_ids.queryItem)
enrichment_df_PUD_PC3_innate = enrichment_df_PUD_PC3_innate[enrichment_df_PUD_PC3_innate['number_of_genes'] >= 5]
if 'Process' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='Process', cmap=cm.cool, title='Up in A1 high samples (PC7)', vmin=0, vmax=0.5, frac_x=False)
if 'KEGG' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='KEGG', cmap=cm.cool, title='Up in A1 high samples (PC7)', vmin=0, vmax=0.5, frac_x=False)
if 'WikiPathways' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='WikiPathways', cmap=cm.cool, title='Up in A1 high samples (PC7)', vmin=0, vmax=0.5, frac_x=False)
if 'Keyword' in enrichment_df_PUD_PC3_innate.value_counts('category').index:
    stringdb_barplot_one_class(df=enrichment_df_PUD_PC3_innate, nb_top=10, clas='Keyword', cmap=cm.cool, title='Up in A1 high samples (PC7)', vmin=0, vmax=0.5, frac_x=False)